# Deploy Notes Gateway (GW2) & OpenSearch

GW2 is the **notes** path: free-text claim-notes backed by an Amazon OpenSearch Serverless collection, fronted by the `opensearch_mcp_server` runtime.

**The GW2 auth mechanism flips by IdP** (`IDP_PROVIDER`):

- **## [OKTA]** — on-behalf-of **token exchange** (RFC 8693) via an AgentCore Identity credential provider. **No interceptor** — identity propagation is native.
- **## [COGNITO]** — a thin **REQUEST interceptor** Lambda (`lakehouse-notes-interceptor`) forwards the caller's `sub`; the gateway→runtime leg uses a Cognito M2M provider.

Both paths enforce the same per-user **row-level security**: the OpenSearch server filters `owner_user_sub` against the caller's subject. The topology is identical to GW1 (DR-1) — only the auth method differs.

**Prerequisites:** `01-deploy-idp.ipynb` (IdP + flag), `04-deploy-mcp-server.ipynb` (claims MCP), `05a-deploy-claims-gateway.ipynb` (claims GW1).

## Prerequisites

- ✅ Run `01-deploy-idp.ipynb` first (sets + persists `IDP_PROVIDER`; on Okta also seeds the test-user subs)
- ✅ Run `04-deploy-mcp-server.ipynb` and `05a-deploy-claims-gateway.ipynb` (claims path)

## What This Notebook Does

1. Creates the OpenSearch Serverless collection (`lakehouse-claim-notes`) — **both paths**
2. Deploys the OpenSearch MCP runtime (`opensearch_mcp_server`; authorizer branched by IdP inside the script) — **both paths**
3. **[COGNITO]** seeds the Cognito test-user subject IDs used for row-level security (Okta subs were seeded in notebook 01)
4. Loads sample claim-note documents — **both paths**
5. Verifies the runtime is `READY` — **both paths**
6. Deploys the GW2 **auth layer** — the flip: **[OKTA]** OBO provider / **[COGNITO]** notes REQUEST interceptor
7. Creates the Notes Gateway (GW2) and its target — the script branches internally (Okta OBO target / Cognito interceptor + M2M target)

## Next Notebook

- **06-deploy-agent.ipynb**

In [ ]:
# AWS Initialization + read the persisted IdP flag (agnostic)
import subprocess
import sys

from utils.notebook_init import init_aws
from utils.idp_config import get_idp_provider

session, AWS_REGION, AWS_ACCOUNT_ID = init_aws()
ssm_client = session.client("ssm", region_name=AWS_REGION)

# Read the flag ONCE from SSM (persisted by notebook 01, Step 0). Every guard
# below branches on this variable — no per-cell SSM re-read.
IDP_PROVIDER = get_idp_provider(ssm_client)

print(f"✅ IDP_PROVIDER = {IDP_PROVIDER}")
print(f"   Account ID: {AWS_ACCOUNT_ID}")
print(f"   Region: {AWS_REGION}")

## Step 1: Create the OpenSearch Serverless Collection

Both paths. Provisions the `lakehouse-claim-notes` collection plus its encryption / network / data-access policies, and stores `opensearch-collection-{arn,endpoint}` in SSM. This must run before the runtime deploy (the runtime role scopes `aoss:APIAccessAll` to this collection ARN).

In [ ]:
subprocess.run(
    [sys.executable, "01_deploy_opensearch_collection.py"],
    cwd="deployment/5b-obo-gateway-setup",
    check=True,
)

## Step 2: Deploy the OpenSearch MCP Runtime

Both paths. Deploys the `opensearch_mcp_server` runtime (reads the collection ARN + `IDP_PROVIDER`; the JWT authorizer is branched inside the script — Cognito M2M `allowedClients` vs Okta `allowedAudience`). Builds a container, so this can take a few minutes.

In [ ]:
subprocess.run(
    [sys.executable, "deploy_runtime.py"],
    cwd="deployment/4b-mcp-opensearch-server",
    check=True,
)

## Step 3: Seed User Subject IDs for Row-Level Security

**## [COGNITO] only.** The sample-data loader (Step 4) tags each note with an `owner_user_sub` taken from the seeded `*-user-*-sub` SSM keys, and **fails if none exist**. On Cognito those keys are written here (`seed_cognito_user_subs.py` reads each test user's `sub`). On Okta they were already written by `setup_okta` in notebook 01, so this step is skipped.

> Sequenced **before** Step 4 by dependency: the loader is non-vacuous only if the subject IDs are already in SSM.

In [ ]:
if IDP_PROVIDER == "cognito":
    subprocess.run(
        [sys.executable, "seed_cognito_user_subs.py"],
        cwd="deployment/4b-mcp-opensearch-server",
        check=True,
    )
else:
    print("⏭️  Skipped: Cognito-only. Okta test-user subs were seeded by notebook 01 (setup_okta).")

## Step 4: Load Sample Claim Notes

Both paths. Bulk-loads disjoint free-text claim-note documents, one slice per test user, tagging each with the user's `owner_user_sub`. Depends on the **collection** (Step 1) and the **seeded subs** (Step 3 / notebook 01) — not on the runtime, since it signs to OpenSearch directly.

In [ ]:
subprocess.run(
    [sys.executable, "load_sample_opensearch_data.py"],
    cwd="deployment/4b-mcp-opensearch-server",
    check=True,
)

## Step 5: Verify the Runtime

Both paths. Read-only health check: confirms the runtime SSM keys are present and `GetAgentRuntime` reports `READY`.

In [ ]:
subprocess.run(
    [sys.executable, "02_verify_opensearch_mcp.py"],
    cwd="deployment/5b-obo-gateway-setup",
    check=True,
)

## Step 6: Deploy the GW2 Auth Layer

This is the **auth-flip**. Only the cell matching `IDP_PROVIDER` runs; the other prints a skip.

### [OKTA] — OBO token-exchange provider

Creates the AgentCore Identity credential provider `lakehouse-obo-okta-provider` (vendor `CustomOauth2`, `TOKEN_EXCHANGE` grant). This is the RFC 8693 substrate GW2 uses to propagate user identity — no interceptor.

In [ ]:
if IDP_PROVIDER == "okta":
    subprocess.run(
        [sys.executable, "03_create_oauth_provider.py"],
        cwd="deployment/5b-obo-gateway-setup",
        check=True,
    )
else:
    print("⏭️  Skipped: OBO provider is [OKTA]-only (IDP_PROVIDER=cognito).")

### [COGNITO] — Notes REQUEST interceptor

Deploys the `lakehouse-notes-interceptor` Lambda (thin REQUEST interceptor that injects the caller's Cognito `sub` on the body-context channel) and stores its ARN in SSM for the gateway create step.

In [ ]:
if IDP_PROVIDER == "cognito":
    subprocess.run(
        ["bash", "deploy.sh"],
        cwd="deployment/5a-gateway-setup/interceptor-notes",
        check=True,
    )
else:
    print("⏭️  Skipped: notes REQUEST interceptor is [COGNITO]-only (IDP_PROVIDER=okta).")

## Step 7: Create the Notes Gateway (GW2)

Both paths. `04_create_obo_gateway.py` branches internally on `IDP_PROVIDER`: **[OKTA]** builds the OBO `TOKEN_EXCHANGE` target; **[COGNITO]** builds the interceptor gateway + Cognito M2M `client_credentials` target. The **DR-11 pre-flight guard** runs here — it refuses to converge a gateway that was deployed for the other IdP (run teardown first). Stores `notes-gateway-{id,arn,url,name}` in SSM.

In [ ]:
subprocess.run(
    [sys.executable, "04_create_obo_gateway.py"],
    cwd="deployment/5b-obo-gateway-setup",
    check=True,
)

## Agent OBO grant — note only

The agent deliberately holds **NO** OBO grant. In the gateway-mediated OBO flow the GW2 gateway's own role performs the RFC 8693 exchange, so the agent role never needs `bedrock-agentcore:GetWorkloadAccessTokenForJWT` (Finding 15). There is no agent-IAM patch step in the happy path.

## Summary

✅ **Notes Gateway (GW2) & OpenSearch deployment complete!**

**What was created (both paths):**
- OpenSearch Serverless collection `lakehouse-claim-notes` (+ encryption/network/data policies)
- `opensearch_mcp_server` runtime (IdP-branched authorizer)
- Sample claim-note documents with per-user `owner_user_sub`
- Notes Gateway (GW2) `lakehouse-notes-gateway` + target

**GW2 auth (the flip):**
- **[OKTA]** — OBO token exchange (RFC 8693) via `lakehouse-obo-okta-provider`; no interceptor
- **[COGNITO]** — `lakehouse-notes-interceptor` REQUEST interceptor + Cognito M2M provider; caller subs seeded for RLS

All configuration saved to SSM Parameter Store.

**Next Steps:**
Run **06-deploy-agent.ipynb** — the agent wires two MCP clients (`claims/` → GW1, `notes/` → GW2).